# 253. Meeting Rooms II
**Difficulty:** 🟡 Medium (Premium) · **Topic:** Interval · **LeetCode:** https://leetcode.com/problems/meeting-rooms-ii/

## 💡 Concepts

**Core concept(s):** Track how many meetings run at once — with a **min-heap** of end times, or a **sweep line**.

**Why it applies here:** The number of rooms needed is the maximum number of meetings overlapping at any instant. A heap of end times reuses a room when a meeting ends before the next starts; a sweep line counts starts (+1) and ends (-1) over time.

**Key intuition:** Rooms needed = the most meetings happening at the same time.

---

### 📚 Sorting to Reveal Order
Sorting intervals by their start (or end) puts overlapping ones next to each other, so a single left-to-right pass can merge or count them. Cost: **O(n log n)**.

### 📚 What is a Heap (Priority Queue)?
A **heap** always gives its smallest (min-heap) or largest (max-heap) item in **O(log n)**. Ideal for "keep the top K" or "always grab the current extreme".
- **In Python:** `heapq` (a min-heap; negate values for a max-heap).

---

**Prerequisite knowledge:**
- Sorting.
- A min-heap of end times, or a start/end sweep.

## 📝 Problem

Return the minimum number of meeting rooms required.

**Example**
```
[[0,30],[5,10],[15,20]] -> 2
```

> Two approaches, both `O(n log n)`: min-heap of end times and a sweep line.

### Approach 1 — Min-Heap of End Times

**Idea:** Sort by start. Keep a heap of end times of ongoing meetings. For each meeting, if the earliest end ≤ its start, reuse that room (pop); always push this meeting's end. The heap size is the rooms in use.

**Time:** `O(n log n)`. **Space:** `O(n)`.

In [ ]:
import heapq

def min_meeting_rooms_heap(intervals):
    if not intervals:
        return 0
    intervals.sort(key=lambda x: x[0])     # process meetings in start order
    heap = []                              # end times of meetings currently using a room
    for start, end in intervals:
        if heap and heap[0] <= start:      # the earliest-ending meeting is already over
            heapq.heappop(heap)            # -> free up (reuse) that room
        heapq.heappush(heap, end)          # this meeting occupies a room until `end`
    return len(heap)                       # rooms in use at the peak = heap size

### Approach 2 — Sweep Line

**Idea:** Sort start times and end times separately. Walk time forward: a start needs a room (+1), an end frees one. The peak count is the answer.

**Time:** `O(n log n)`. **Space:** `O(n)`.

In [ ]:
def min_meeting_rooms_sweep(intervals):
    starts = sorted(i[0] for i in intervals)   # all start times
    ends = sorted(i[1] for i in intervals)     # all end times
    rooms = peak = 0
    s = e = 0
    while s < len(starts):
        if starts[s] < ends[e]:            # a meeting starts before the next one ends
            rooms += 1; s += 1             # -> need one more room
            peak = max(peak, rooms)        # track the maximum concurrent meetings
        else:
            rooms -= 1; e += 1             # a meeting ended -> a room frees up
    return peak

In [ ]:
# Correctness check
tests = [([[0,30],[5,10],[15,20]],2), ([[7,10],[2,4]],1), ([],0), ([[1,5],[2,6],[3,7]],3)]
for iv, exp in tests:
    a = min_meeting_rooms_heap([x[:] for x in iv])
    b = min_meeting_rooms_sweep([x[:] for x in iv])
    print(f"{iv} -> heap={a}, sweep={b} | expected={exp}")
    assert a == b == exp, "mismatch!"
print("\nAll tests passed")

## ⏱️ Empirically Checking the Complexities

We time each approach on inputs of growing size `n` and read the **doubling ratio**.

| Theoretical | Ratio `n`→`2n` |
|---|---|
| `O(n)`       | ≈ **2×** |
| `O(n log n)` | ≈ **2×** (slightly more) |
| `O(n²)`     | ≈ **4×** |

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(6):
    if os.path.exists(os.path.join(_root, "bench_utils.py")): break
    _root = os.path.dirname(_root)
if _root not in sys.path: sys.path.insert(0, _root)
from bench_utils import benchmark

def make_worst_case(n):
    intervals = [[0, i + 1] for i in range(n)]   # all overlap at time 0 -> n rooms
    return (intervals,)
solutions = {
    "heap  O(n log n)": min_meeting_rooms_heap,
    "sweep O(n log n)": min_meeting_rooms_sweep,
}
sizes = [20000, 40000, 80000, 160000]

benchmark(solutions, make_worst_case, sizes, plot=True)


## 🧩 Patterns Learned

- **Max concurrent = resource count:** rooms needed equals the peak overlap.
- **Two tools:** a heap of end times, or a start/end sweep line — both classic.
- **Signal:** "minimum rooms / servers / platforms", "max overlap at once".
- **Related problems:** Meeting Rooms, Car Pooling, My Calendar III.
- **Common pitfalls:** (1) comparing starts and ends with the wrong strictness; (2) not tracking the peak (only the final count).